# Postprocessor for CAGMDP / TriCycLe

In [1]:
import pickle
import networkx as nx
import numpy as np
import os

CAGMDP_DIR = "./CAGMDP_result/"
TRICYCLE_DIR = "./TriCycLe_result/"

#```<dataset>_[method]_w_triangles_<epsilon>_final_dp_<serialno>.txt```
#```<dataset>_[method]_attribute_<epsilon>_<serialno>.txt```

DATAS = ["Brightkite"]#, ["LastFM", "GitHub"]
METHODS = [""] # ["", "TriCycle"] Note "" = CAGMDP
EPS = [0.5, 0.75, 1, 1.5, 2, 3, 4.5, 6.5, 9, 12, 16, 20]
N_TRIALS = 10

for dataset in DATAS:
    for method, curr_dir in zip(METHODS, [CAGMDP_DIR]):
        out_dir = os.path.join(curr_dir, dataset)
        #directory = "_".join(word for word in [dataset, method, "combined", "experiment"] if word)
        for i in range(N_TRIALS):
            directory = os.path.join(curr_dir, f"txts_{dataset}_trial_{i}")
            for eps in EPS:
                # adjacency
                filename = "_".join(word for word in [dataset, method, "w_triangles", str(float(eps)), "final_dp_0.txt"] if word)
                with open(os.path.join(directory, filename), 'r') as f:
                    e = []
                    for line in f.readlines():
                        v, w = line.split()
                        e.append((int(v), int(w)))
                    g = nx.Graph(e)
                with open(os.path.join(out_dir, f"nx_adj_eps{eps}_i{i}.pkl"), 'wb') as f:
                    pickle.dump(g, f)
                # attribute
                filename = "_".join(word for word in [dataset, method, "attribute", str(float(eps)), "0.txt"] if word)
                with open(os.path.join(directory, filename), 'r') as f:
                    n = max([int(line.split()[0]) for line in f.readlines()]) + 1                
                with open(os.path.join(directory, filename), 'r') as f:
                    m = max([len(line.split())-1 for line in f.readlines()])
                a = np.zeros((n,m))
                with open(os.path.join(directory, filename), 'r') as f:
                    for row in f.readlines():
                        full_col = row.split()
                        row_id = full_col[0]
                        for col_id, data in enumerate(full_col[1:]):
                            a[int(row_id)][int(col_id)] = int(data)
                np.savetxt(os.path.join(out_dir, f"np_att_eps{eps}_i{i}.txt"), a, fmt="%d")
                